In [0]:
import time
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS movie_recommender.silver

In [0]:
link_df=spark.read.table('movie_recommender.bronze.links')

link_df=link_df.withColumn('movieId',F.col('movieId').cast('int')).withColumn('imdbId',F.col('imdbId').cast('string')).withColumn('tmdbId',F.col('tmdbId').cast('int')).where(F.col('tmdbId').isNotNull())

link_df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('movie_recommender.silver.links')

In [0]:
tags_df=spark.read.table('movie_recommender.bronze.tags')

tags_df=tags_df.withColumn('userId',F.col('userId').cast('int')).withColumn('movieId',F.col('movieId').cast('int')).withColumn('tag',F.col('tag').cast('string')).withColumn('tag_ts',F.timestamp_seconds(F.col('timestamp').cast('long'))).drop('timestamp')

tags_df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('movie_recommender.silver.tags')

In [0]:
movies_df=spark.read.format('delta').table('movie_recommender.bronze.movies')

movies_df=movies_df.withColumn('movieId',F.col('movieId').cast('int')).withColumn('title',F.lower(F.trim(F.col('title')))).withColumn('genre',F.explode(F.split(F.col('genres'),F.lit('\\|')))).withColumn('release_year',F.regexp_extract(F.col('title'),'\\((\\d+)\\)',1)).drop('genres')

movies_df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('movie_recommender.silver.movies')

In [0]:
ratings_df=spark.read.format('delta').table('movie_recommender.bronze.ratings')

ratings_df=ratings_df.withColumn('userId',F.col('userId').cast('int')).withColumn('movieId',F.col('movieId').cast('int')).withColumn('rating',F.col('rating').cast('float')).withColumn('rating_ts',F.timestamp_seconds(F.col('timestamp').cast('long'))).withColumn('year',F.year(F.col('rating_ts'))).withColumn('month',F.month(F.col('rating_ts'))).drop('timestamp')

ratings_df.write.format('delta').mode('overwrite').option('mergeSchema','true').partitionBy('year').saveAsTable('movie_recommender.silver.ratings')

In [0]:
tmdb_schema=StructType([
    StructField('id',IntegerType(),True),
    StructField('title',StringType(),True),
    StructField('overview',StringType(),True),
    StructField('release_date',StringType(),True),
    StructField('runtime',IntegerType(),True),
    StructField('popularity',FloatType(),True),
    StructField('vote_average',FloatType(),True),
    StructField('vote_count',IntegerType(),True),
    StructField('genres',ArrayType(StringType()),True),
    StructField('spoken_languages',ArrayType(StringType()),True),
    StructField('production_countries',ArrayType(StringType()),True),    
    StructField('tagline',StringType(),True)
])

In [0]:
tmdb_metadata = set()

if spark.catalog.tableExists('movie_recommender.silver.tmdb_metadata'):
    tmdb_metadata_df=spark.read.format('delta').table('movie_recommender.silver.tmdb_metadata')
    tmdb_metadata={row['id'] for row in tmdb_metadata_df.select('id').collect() if 'id' in row.asDict()}

batch=[]

tmdb_id=[row['tmdbId'] for row in link_df.select('tmdbId').collect()]

tmdb_id=[i for i in tmdb_id if i not in tmdb_metadata]

for id in tmdb_id:
    try:
        req=tmdb.Movies(id)
        response=req.info()
        batch.append({'id':req.id,'title':req.title,'overview':req.overview,'release_date':req.release_date,'runtime':req.runtime,'popularity':req.popularity,'vote_average':req.vote_average,'vote_count':req.vote_count,'genres':[g['name'] for g in req.genres],'spoken_languages':[l['name'] for l in req.spoken_languages],'production_countries':[c['name'] for c in req.production_countries],'tagline':req.tagline})
        if len(batch)==500:
            tmdb_df=spark.createDataFrame(batch,tmdb_schema)
            tmdb_df.write.format('delta').mode('append').saveAsTable('movie_recommender.silver.tmdb_metadata')
            batch=[]
        time.sleep(0.05)
    except Exception as e:
        print(e)  
tmdb_df=spark.createDataFrame(batch,tmdb_schema)
tmdb_df.write.format('delta').mode('append').saveAsTable('movie_recommender.silver.tmdb_metadata')